## Loading Data

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
gpu_list = tf.config.experimental.list_physical_devices('GPU')
num_gpus = len(gpu_list)
if len(gpu_list) > 0:
    for gpu in gpu_list:
        tf.config.experimental.set_memory_growth(gpu, True)
        
from waymo_open_dataset.protos import scenario_pb2
from torch_geometric.loader import DataLoader
from datasets import WaymoDataset


val_dataset = WaymoDataset(
    data_root='./data/waymo/processed/',
    split_dir='processed_scenarios_validation',
    info_file='processed_scenarios_validation_infos.pkl',
)

val_dataloader = DataLoader(
    val_dataset, batch_size=1, num_workers=1,
    pin_memory=True, persistent_workers=True,
    shuffle=False,
)

val_dataloader_iter = iter(val_dataloader)

In [ ]:
data = next(val_dataloader_iter)
print('scenario:', data['scenario_id'][0])

tf_dataset = tf.data.TFRecordDataset(os.path.join(
    './data/waymo/proto_v1.1/validation/',
    data['tfrecord_file'][0]
)).skip(data['tfrecord_idx'][0]).take(1)
tf_dataset_iter = tf_dataset.as_numpy_iterator()
scenario = scenario_pb2.Scenario.FromString(next(tf_dataset_iter))
assert scenario.scenario_id == data['scenario_id'][0]

## Inference

In [ ]:
import math
import torch
from tqdm import tqdm
from models import UniMM


model = UniMM.load_from_checkpoint(
    checkpoint_path='/path/to/your/ckpt',
    map_location='cpu',
).eval().cuda()

joint_scenes = []
data_cuda = data.to(model.device)
for _ in tqdm(range(model.num_eval_rollouts)):
    with torch.inference_mode():
        sim_dict = model.inference(
            data_cuda,
            sim_steps=math.ceil(model.eval_future_steps/model.num_execution_steps),
        )
    
    sim_positions = sim_dict['sim_positions'][:, :model.eval_future_steps]
    sim_headings = sim_dict['sim_headings'][:, :model.eval_future_steps]
    current_z = data['agent']['position'][:, model.num_historical_steps - 1, 2]

    traj_eval = torch.cat([
        sim_positions,
        current_z.unsqueeze(1).unsqueeze(-1).repeat(1, model.eval_future_steps, 1),
        sim_headings.unsqueeze(-1),
    ], dim=-1)

    joint_scenes.append(traj_eval)


In [ ]:
import numpy as np
from waymo_open_dataset.utils.sim_agents import submission_specs
from waymo_open_dataset.protos import sim_agents_submission_pb2

def joint_scene_from_states(states, object_ids) -> sim_agents_submission_pb2.JointScene:
    # States shape: (num_objects, num_steps, 4).
    # Objects IDs shape: (num_objects,).
    simulated_trajectories = []
    for i_object in range(len(object_ids)):
        simulated_trajectories.append(sim_agents_submission_pb2.SimulatedTrajectory(
            center_x=states[i_object, :, 0], center_y=states[i_object, :, 1],
            center_z=states[i_object, :, 2], heading=states[i_object, :, 3],
            object_id=object_ids[i_object]
        ))
    return sim_agents_submission_pb2.JointScene(
        simulated_trajectories=simulated_trajectories)

def scenario_rollouts_from_states(
    states, object_ids, scenario_id
) -> sim_agents_submission_pb2.ScenarioRollouts:
    # States shape: (num_rollouts, num_objects, num_steps, 4).
    # Objects IDs shape: (num_objects,).
    joint_scenes = []
    for i_rollout in range(states.shape[0]):
        joint_scenes.append(joint_scene_from_states(states[i_rollout], object_ids))
    return sim_agents_submission_pb2.ScenarioRollouts(
        # Note: remember to include the Scenario ID in the proto message.
        joint_scenes=joint_scenes, scenario_id=scenario_id)

object_ids_all = data['agent']['id'][0]
states_all = torch.stack(joint_scenes, dim=0)
object_ids = submission_specs.get_sim_agent_ids(scenario)
states = states_all[:, np.isin(object_ids_all, object_ids)]
scenario_rollouts = scenario_rollouts_from_states(states, object_ids, scenario.scenario_id)
submission_specs.validate_scenario_rollouts(scenario_rollouts, scenario)

## Visualization

In [ ]:
simulated_states_list = []
for joint_scene in scenario_rollouts.joint_scenes:
    simulated_agent_list = []
    for agent in joint_scene.simulated_trajectories:
        simulated_agent = tf.stack([
            agent.center_x, agent.center_y,
            agent.center_z, agent.heading
        ], axis=-1)
        simulated_agent_list.append(simulated_agent)
    simulated_states_list.append(tf.stack(simulated_agent_list, axis=0))
simulated_states = tf.stack(simulated_states_list, axis=0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
from waymo_open_dataset.utils import trajectory_utils
from waymo_open_dataset.utils.sim_agents import visualizations


ROLLOUT_INDEX = 0

logged_trajectories = trajectory_utils.ObjectTrajectories.from_scenario(scenario)
logged_trajectories = logged_trajectories.gather_objects_by_id(
    tf.convert_to_tensor(submission_specs.get_sim_agent_ids(scenario))
)
n_objects = logged_trajectories.valid.shape[0]
lengths = tf.broadcast_to(
    logged_trajectories.length[:, submission_specs.CURRENT_TIME_INDEX, tf.newaxis], 
    (n_objects, submission_specs.N_SIMULATION_STEPS)
)
widths = tf.broadcast_to(
    logged_trajectories.width[:, submission_specs.CURRENT_TIME_INDEX, tf.newaxis],
    (n_objects, submission_specs.N_SIMULATION_STEPS)
)

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
fig.tight_layout()
ax.set_xticks([])
ax.set_yticks([])

# evaluated agents are Purple (4), other agents are blue (0)
color_idx=np.zeros((simulated_states.shape[1], simulated_states.shape[2]), dtype=np.int32)
eval_ids = submission_specs.get_evaluation_sim_agent_ids(scenario)
object_ids = submission_specs.get_sim_agent_ids(scenario)
for i, agent_id in enumerate(object_ids):
    if agent_id in eval_ids:
        color_idx[i] = 4

anim = visualizations.get_animated_states(
    fig, ax, scenario,
    x=simulated_states[ROLLOUT_INDEX, :, :, 0],
    y=simulated_states[ROLLOUT_INDEX, :, :, 1],
    yaw=simulated_states[ROLLOUT_INDEX, :, :, 3],
    length=lengths, width=widths,
    color_idx=color_idx,
)

HTML(anim.to_jshtml())

## WOSAC Metrics

In [ ]:
from google.protobuf import text_format
from distutils.sysconfig import get_python_lib
from waymo_open_dataset.protos import sim_agents_metrics_pb2
from waymo_open_dataset.wdl_limited.sim_agents_metrics import metrics

config_path = f'{get_python_lib()}/waymo_open_dataset/wdl_limited/sim_agents_metrics/challenge_2024_config.textproto'
with open(config_path, 'r') as f:
    config = sim_agents_metrics_pb2.SimAgentMetricsConfig()
    text_format.Parse(f.read(), config)

scenario_metrics = metrics.compute_scenario_metrics_for_bundle(
    config, scenario, scenario_rollouts
)
print(scenario_metrics)